# Dynamic JSCC — figures from the GitHub repository

This notebook does **not** contain a copy of the model. It clones
[`quantum-dynamic-jscc`](https://github.com/rowshan-mannan-oni/quantum-dynamic-jscc) and runs that repository's own scripts, so the figures always
reflect the latest committed code.

**What you need before running**

| Requirement | How |
|---|---|
| GPU | Settings -> Accelerator -> GPU (T4/P100) |
| Internet | Settings -> Internet -> **On** (needed to clone the repo) |
| Trained weights | Add your checkpoint as a Kaggle Dataset (see below) |
| CIFAR-10 | Add the *CIFAR-10 Python* dataset, or let it download |

The weights are **not** stored in the repository (they are gitignored). Upload the
notebook checkpoint - the file containing `{'SE','CE','G','P'}`, e.g. `final.pth` - as a
Kaggle Dataset and attach it via **Add Input**. The cells below find it automatically.

If you have no trained weights yet, run the repository's `train_dyna.py` instead;
the figure step here expects an existing checkpoint.

## 1. Clone (or update) the repository

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/rowshan-mannan-oni/quantum-dynamic-jscc.git"
REPO_DIR = "/kaggle/working/quantum-dynamic-jscc"

if os.path.isdir(REPO_DIR):
    print("Repository present - pulling latest commit")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("\nWorking directory:", os.getcwd())
subprocess.run(["git", "log", "-1", "--format=Using commit %h - %s"], check=True)

## 2. Dependencies

Kaggle already ships a CUDA build of PyTorch, so **do not** install the pinned torch from
`requirements.txt` here - that would replace a working GPU install. Only the two plotting
and metric packages are needed, and they are usually present already.

In [ ]:
import importlib, subprocess, sys

for module, package in [("skimage", "scikit-image"), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(module) is None:
        print(f"installing {package} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
    else:
        print(f"{package} already available")

import torch
print("\ntorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Enable it under Settings -> Accelerator.")

## 3. CIFAR-10

Looks for an attached Kaggle CIFAR-10 dataset first (no download, works with Internet off
for this step). Falls back to downloading, which needs Internet on.

In [ ]:
import glob, os, shutil, tarfile

DATA_ROOT = "/kaggle/working/data"
os.makedirs(DATA_ROOT, exist_ok=True)
target = os.path.join(DATA_ROOT, "cifar-10-batches-py")

def find(pattern):
    hits = glob.glob(pattern, recursive=True)
    return hits[0] if hits else None

if not os.path.isdir(target):
    folder = find("/kaggle/input/**/cifar-10-batches-py")
    tar = find("/kaggle/input/**/cifar-10-python.tar.gz")
    if folder:
        print("Found extracted dataset:", folder)
        shutil.copytree(folder, target)
    elif tar:
        print("Found tarball:", tar)
        with tarfile.open(tar) as t:
            t.extractall(DATA_ROOT)
    else:
        print("No CIFAR-10 in /kaggle/input - it will be downloaded (Internet must be On).")

print("Ready:", os.path.isdir(target))

## 4. Locate the weights and convert them

The repository stores each sub-network in its own file, while the notebook checkpoint packs
all four into one dict. `convert_kaggle_weights.py` translates between the two and validates
every state dict with `strict=True` before writing.

**The settings below must match the run that produced the weights** - they also determine the
checkpoint folder name that the figure script looks in.

In [ ]:
import glob, subprocess, sys, torch

# ---- must match the training run that produced the weights ----
LAMBDA_REWARD = 1.5e-3
SELECT        = "hard"
C_CHANNEL     = 16

def looks_like_checkpoint(path):
    try:
        ck = torch.load(path, map_location="cpu")
        return isinstance(ck, dict) and {"SE", "CE", "G", "P"}.issubset(ck.keys())
    except Exception:
        return False

candidates = sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True))
weights = next((p for p in candidates if looks_like_checkpoint(p)), None)

if weights is None:
    raise SystemExit(
        "No checkpoint containing {'SE','CE','G','P'} found under /kaggle/input.\n"
        "Upload your final.pth as a Kaggle Dataset and attach it with Add Input.\n"
        f"Files scanned: {candidates}")

print("Using weights:", weights)
subprocess.run([sys.executable, "convert_kaggle_weights.py",
                "--src", weights,
                "--lambda_reward", str(LAMBDA_REWARD),
                "--select", SELECT,
                "--C_channel", str(C_CHANNEL),
                "--force"], check=True)

## 5. Generate the figures

`make_figures.py` runs the SNR sweep, the fixed-rate sweep, the per-class breakdown and the
sample reconstructions, writing both PNGs and the CSVs behind them.

Reduce `--num_test` for a quick pass; 10000 uses the whole test set.

In [ ]:
import subprocess, sys

NUM_TEST = 10000        # lower this (e.g. 1000) for a fast check
SNR_LIST = "0,2,4,6,8,10,12,14,16,18,20"

subprocess.run([sys.executable, "-u", "make_figures.py",
                "--gpu_ids", "0" if torch.cuda.is_available() else "-1",
                "--select", SELECT,
                "--lambda_reward", str(LAMBDA_REWARD),
                "--C_channel", str(C_CHANNEL),
                "--num_test", str(NUM_TEST),
                "--snr_list", SNR_LIST,
                "--fig6_snr", "10",
                "--num_workers", "2",
                "--dataroot", DATA_ROOT,
                "--results_dir", "/kaggle/working/figures"], check=True)

## 6. Show the figures

In [ ]:
from IPython.display import Image, display, Markdown
import os

FIG_DIR = "/kaggle/working/figures"
titles = {
    "fig4_rate_psnr_vs_snr.png": "Average rate (CPP) and PSNR vs SNR",
    "fig5_attainable_psnr.png":  "Attainable PSNR: fixed rates vs adaptive",
    "fig6_per_class.png":        "Per-class rate and PSNR",
    "reconstructions.png":       "Sample reconstructions",
}
for name, title in titles.items():
    path = os.path.join(FIG_DIR, name)
    if os.path.exists(path):
        display(Markdown(f"### {title}"))
        display(Image(filename=path))
    else:
        print("missing:", path)

In [ ]:
import pandas as pd, os

for csv_name in ["fig4_data.csv", "fig5_data.csv", "fig6_data.csv"]:
    path = os.path.join(FIG_DIR, csv_name)
    if os.path.exists(path):
        print(f"\n===== {csv_name} =====")
        display(pd.read_csv(path))

## Notes

- Figures and CSVs are written to `/kaggle/working/figures` and can be downloaded from the
  notebook's **Output** tab.
- Re-running cell 1 pulls the newest commit, so this notebook tracks the repository. Restart
  the kernel after a pull if the model code changed, so Python re-imports it.
- To evaluate a different operating point, retrain with another `lambda_reward`, upload those
  weights, and change `LAMBDA_REWARD` in cell 4 - it selects the checkpoint folder.
- `test_dyna.py` in the repository prints scalar metrics (PSNR, SSIM, per-class rates) if you
  want numbers without plots.